
# Hospital EDW — Attribute-Based Access Control (ABAC) Tag Policies

This notebook implements **Approach 1 (Centralized ABAC Tag Policies)** in Databricks Unity Catalog using `CREATE POLICY`.

### Why Approach 1:
1. **Centralized Enforcement**: Policies attach at the `CATALOG` level (`hospital_catalog`).
2. **Automated Protection**: Any column tagged with `info_type` or `financial = true` is automatically masked for restricted groups.
3. **Zero Maintenance**: New tables added to the catalog are governed immediately upon tagging.


## Step 1: Create Governance Schema & Masking Functions

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS hospital_catalog.governance
COMMENT 'Central governance schema for ABAC policy UDFs and access control rules';

In [ ]:
%sql
-- 1.1 Name Masking UDF (Patients & Doctors)
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_name(val STRING)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN val
  WHEN val IS NULL THEN NULL
  ELSE CONCAT(SUBSTRING(val, 1, 1), '****')
END;

In [ ]:
%sql
-- 1.2 Phone Number Masking UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_phone(phone STRING)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN phone
  WHEN phone IS NULL THEN NULL
  ELSE CONCAT('***-***-', RIGHT(phone, 4))
END;

In [ ]:
%sql
-- 1.3 Physical Address Masking UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_address(address STRING)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN address
  WHEN address IS NULL THEN NULL
  ELSE '*** REDACTED ADDRESS ***'
END;

In [ ]:
%sql
-- 1.4 Date of Birth Masking UDF (HIPAA Safe Harbor)
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_dob(dob DATE)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN dob
  WHEN dob IS NULL THEN NULL
  ELSE MAKE_DATE(YEAR(dob), 1, 1)
END;

In [ ]:
%sql
-- 1.5 Financial Amount Masking UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_financial(amount DOUBLE)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('billing_finance')
    THEN amount
  ELSE NULL
END;

In [ ]:
%sql
-- 1.6 Hospital Branch Isolation Row Filter UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.filter_branch(branch STRING)
RETURN 
  is_account_group_member('hospital_admins') OR
  (is_account_group_member('cairo_branch_staff') AND branch = 'CAI') OR
  (is_account_group_member('alexandria_branch_staff') AND branch = 'ALX') OR
  (is_account_group_member('giza_branch_staff') AND branch = 'GIZ');


## Step 2: Create Centralized ABAC Tag Policies

In [ ]:
%sql
-- Policy 1: Patient & Doctor Names
CREATE OR REPLACE POLICY pii_name_mask_policy
ON CATALOG hospital_catalog
COLUMN MASK hospital_catalog.governance.mask_name
TO `data_analysts`, `researchers`
EXCEPT `hospital_admins`, `clinical_stewards`
FOR COLUMNS
MATCH COLUMNS 
  has_tag_value('info_type', 'first_name') OR
  has_tag_value('info_type', 'last_name') OR
  has_tag_value('info_type', 'staff_name');

In [ ]:
%sql
-- Policy 2: Phone Numbers
CREATE OR REPLACE POLICY pii_phone_mask_policy
ON CATALOG hospital_catalog
COLUMN MASK hospital_catalog.governance.mask_phone
TO `data_analysts`, `researchers`
EXCEPT `hospital_admins`, `clinical_stewards`
FOR COLUMNS
MATCH COLUMNS 
  has_tag_value('info_type', 'phone_number') OR
  has_tag_value('info_type', 'business_phone_number');

In [ ]:
%sql
-- Policy 3: Physical Address
CREATE OR REPLACE POLICY pii_address_mask_policy
ON CATALOG hospital_catalog
COLUMN MASK hospital_catalog.governance.mask_address
TO `data_analysts`, `researchers`
EXCEPT `hospital_admins`, `clinical_stewards`
FOR COLUMNS
MATCH COLUMNS has_tag_value('info_type', 'physical_address');

In [ ]:
%sql
-- Policy 4: Date of Birth (HIPAA Safe Harbor)
CREATE OR REPLACE POLICY phi_dob_mask_policy
ON CATALOG hospital_catalog
COLUMN MASK hospital_catalog.governance.mask_dob
TO `data_analysts`, `researchers`
EXCEPT `hospital_admins`, `clinical_stewards`
FOR COLUMNS
MATCH COLUMNS has_tag_value('info_type', 'date_of_birth');

In [ ]:
%sql
-- Policy 5: Financial Monetary Amounts
CREATE OR REPLACE POLICY financial_amount_mask_policy
ON CATALOG hospital_catalog
COLUMN MASK hospital_catalog.governance.mask_financial
TO `clinical_stewards`, `data_analysts`, `researchers`
EXCEPT `hospital_admins`, `billing_finance`
FOR COLUMNS
MATCH COLUMNS has_tag_value('financial', 'true');

In [ ]:
%sql
-- Policy 6: Branch Data Isolation Row Filter
CREATE OR REPLACE POLICY branch_isolation_row_policy
ON CATALOG hospital_catalog
ROW FILTER hospital_catalog.governance.filter_branch(branch => source_branch)
TO `cairo_branch_staff`, `alexandria_branch_staff`, `giza_branch_staff`
EXCEPT `hospital_admins`
FOR TABLES
MATCH COLUMNS column_name = 'source_branch' AS source_branch;


## Step 3: Grant Table Privileges to Groups

In [ ]:
%sql
-- 3.1 Data Analysts (BI Developers, Power BI, Tableau)
GRANT USAGE ON CATALOG hospital_catalog TO `data_analysts`;
GRANT USAGE ON SCHEMA hospital_catalog.gold TO `data_analysts`;
GRANT SELECT ON ALL TABLES IN SCHEMA hospital_catalog.gold TO `data_analysts`;
REVOKE ALL PRIVILEGES ON SCHEMA hospital_catalog.silver FROM `data_analysts`;

In [ ]:
%sql
-- 3.2 Clinical Stewards & Clinicians
GRANT USAGE ON CATALOG hospital_catalog TO `clinical_stewards`;
GRANT USAGE ON SCHEMA hospital_catalog.silver TO `clinical_stewards`;
GRANT USAGE ON SCHEMA hospital_catalog.gold TO `clinical_stewards`;
GRANT SELECT ON ALL TABLES IN SCHEMA hospital_catalog.gold TO `clinical_stewards`;
GRANT SELECT ON TABLE hospital_catalog.silver.patient_steward_review_queue_enterprise TO `clinical_stewards`;
GRANT SELECT ON TABLE hospital_catalog.silver.doctor_steward_review_queue_enterprise TO `clinical_stewards`;

In [ ]:
%sql
-- 3.3 Billing & Finance Team
GRANT USAGE ON CATALOG hospital_catalog TO `billing_finance`;
GRANT USAGE ON SCHEMA hospital_catalog.gold TO `billing_finance`;
GRANT SELECT ON TABLE hospital_catalog.gold.fact_billing TO `billing_finance`;
GRANT SELECT ON TABLE hospital_catalog.gold.dim_payer TO `billing_finance`;


## Step 4: Verification Queries

In [ ]:
%sql
-- List all active ABAC policies attached to hospital_catalog
SHOW POLICIES ON CATALOG hospital_catalog;

In [ ]:
%sql
-- Test query
SELECT 
    enterprise_patient_id, 
    first_name, 
    last_name, 
    contact_no, 
    date_of_birth,
    address
FROM hospital_catalog.gold.dim_patient_enterprise
LIMIT 10;